# holdout-data-one-per-class — worked example 3: Holdout gallery preserves multi-dimensional sample shape

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `holdout-data-one-per-class`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

The one-per-class selection pattern works on data of any sample shape, not just flat vectors. For image data of shape `(N, C, H, W)`, `data[mask][0]` produces a single sample of shape `(C, H, W)`, and `t.stack(per_class, dim=0)` assembles the gallery into shape `(num_classes, C, H, W)`. The selection logic is identical — only the leading batch dimension is collapsed.

## Worked solution

**Step 1 — create a mini image-style dataset.** Simulate `(N, 1, 8, 8)` data with 4 classes — the same shape convention as MNIST (channels, height, width).

**Step 2 — apply the holdout pattern.** The same mask-and-select loop works unchanged regardless of sample dimensionality.

**Step 3 — check output shape.** Expected `(4, 1, 8, 8)` — num_classes as the new leading dimension, original sample shape preserved.

**Step 4 — confirm identity.** For each class, verify the holdout row was actually drawn from a sample with that class label in the original data.

In [ ]:
import torch as t

t.manual_seed(30)

def one_per_class(data: t.Tensor, labels: t.Tensor, num_classes: int) -> t.Tensor:
    per_class = []
    for c in range(num_classes):
        mask = labels == c
        per_class.append(data[mask][0])
    return t.stack(per_class, dim=0)

# Simulate (N, 1, 8, 8) image-style data
t.manual_seed(30)
N, num_classes = 60, 4
C_chan, H, W = 1, 8, 8
data = t.randn(N, C_chan, H, W)
base = t.arange(N) % num_classes
perm = t.randperm(N, generator=t.Generator().manual_seed(7))
data = data[perm]
labels = base[perm]

holdout = one_per_class(data, labels, num_classes)
print(f"Holdout shape: {holdout.shape}")  # (4, 1, 8, 8)
assert holdout.shape == (num_classes, C_chan, H, W)

# Verify each class's sample comes from the right label
for c in range(num_classes):
    class_data = data[labels == c]  # all samples of class c
    # holdout[c] must be the first one
    assert t.allclose(holdout[c], class_data[0]), f"Class {c} holdout mismatch"

print("Shape (4, 1, 8, 8) confirmed; all class identities verified.")